In [ ]:
# Paths come from `paths.py`, never from a literal relative to some checkout.
# That module is the only place that knows where the tree lives, and it names
# the environment variables that override each location (FSCORE_DB above all —
# fscore.db is ~1.7 GB and is not kept in the repository).
# Run this notebook from its own directory, src/fscore_vietnam.
import sys, pathlib

HERE = pathlib.Path.cwd()
assert (HERE / "paths.py").exists(), f"run from src/fscore_vietnam (cwd={HERE})"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from paths import DATA, RESULTS, DB, ensure_dirs, require_db
ensure_dirs()

# Trade turnover — the liquidity filter at portfolio formation

The portfolio is formed on **30 June**. The screen therefore has to describe the stock as it
was on that date, not as it was over the fiscal year the accounting came from:

    window            = the 30 calendar days ending 30 June       (1 - 30 June)
    market calendar   = every date any symbol has a bar on, inside that window
    traded_pct        = sessions with volume > 0 / market calendar sessions
    turnover          = volume over the window / shares outstanding (net of treasury)
    tradeable         = traded_pct >= MIN_TRADED_PCT  and  turnover above the market's
                        TURNOVER_PERCENTILE-th percentile that year

Two screens because they catch different failures. `turnover` asks **how much** could be
bought; `traded_pct` asks **how often** it could be bought at all. A stock whose month of
volume arrives in two sessions passes the first and fails the second.

**Which fiscal year a window screens.** A portfolio formed on 30 June of year *F* trades on
the fiscal year that closed the previous 31 December, so the window of June *F* attaches to
`period = F - 1` (`FORMATION_LAG_YEARS`). Row `(HPG, 2024)` is screened on June 2025 bars.
Set the constant to 0 to screen `period` with its own June instead — that would be a screen
run six months before the accounting it filters was published.

**Denominator = shares outstanding.** Treasury shares cannot trade, so counting them
inflates the denominator and reads the stock as less liquid than it is. The count comes
from `statement_fields.share_count()` — the same rule, the same thresholds and the same
vintage `book_to_market_calculation.ipynb` divides price by, which is what lets
`price_matching_and_finalize.ipynb` assert the two panels agree. `shares_issued` stays in
the panel as the fallback and as the pre-change denominator.

**Volume = `deal_volume` (khớp lệnh).** `putthrough_volume` is carried forward on days with
no block trade in part of the history — 13.4% of all 2017 volume is the same figures
repeated — so `total_volume` cannot be summed. The next section measures it. `turnover_total`
stays in the panel for comparison; the filter does not read it.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

from statement_fields import PAR_VALUE_VND, SHARE_COLS, share_count


FORMATION_MMDD = "06-30"      # portfolio formation date
WINDOW_DAYS = 30              # calendar days of price history ending on that date
FORMATION_LAG_YEARS = 1       # June of year F screens fiscal period F - 1

TURNOVER_PERCENTILE = 10      # drop the bottom decile of the market's cross-section
MIN_TRADED_PCT = 50.0         # and anything trading on under half the window's sessions

## Inputs

Loaded as in `book_to_market_calculation.ipynb`, but with **two** universes, because the
percentile cut is defined on the market and the output is defined on the clean panel:

- `market` — every firm-year in the corrected extract that has charter capital (1,814
  symbols). This is the cross-section the percentile is taken over: *toàn sàn*, as wide as
  the data allows. It is wider than the panel by the firm-years that failed the accounting
  checks but were still listed and trading, and those belong in a liquidity distribution.
- `panel` — the accounting-clean rows, same key as the F-score and BM panels. Only these
  get a `tradeable` verdict.

`shares` is the outstanding count as of `period`'s balance-sheet date, i.e. the **last
figure published** on the formation date — the annual report carrying it lands around March,
three months before the June window. `shares_issued`, the charter capital of the same
`period`, is the fallback and carries the same vintage, so the point-in-time argument is
unchanged by the switch: a capital raise between 1 January and 30 June is in neither, and is
counted in the diagnostics rather than patched.

Both universes go through the same share count. The percentile cut would otherwise be read
off a cross-section built on one denominator and applied to values built on another.

In [2]:
accounting_clean_df = pd.read_csv(f"{RESULTS}/accounting_clean_bol_matrix.csv")
accounting_clean_df.set_index(keys=["symbol", "period"], inplace=True)

extract_df = pd.read_csv(f"{RESULTS}/f_score_fields_extract_corrected.csv")
extract_df.set_index(keys=["symbol", "period"], inplace=True)

market = extract_df[["paid_in_capital"]].sort_index()
panel = extract_df.loc[extract_df.index.isin(accounting_clean_df.index),
                       ["paid_in_capital", "treasury_stock"]].sort_index()

SHARE_SQL = """
select symbol, Year as period, ShareAtPeriodEnd
from fireant_financial_data_general
where Quarter = 0 and ShareAtPeriodEnd > 0
"""

with closing(sqlite3.connect(DB)) as conn:
    vendor_shares = pd.read_sql(SHARE_SQL, conn).set_index(["symbol", "period"])["ShareAtPeriodEnd"]

pd.Series({
    "market universe rows": len(market),
    "market universe symbols": market.index.get_level_values("symbol").nunique(),
    "panel rows": len(panel),
    "panel symbols": panel.index.get_level_values("symbol").nunique(),
    "vendor share counts loaded": len(vendor_shares),
})

market universe rows          23493
market universe symbols        1814
panel rows                    21233
panel symbols                  1805
vendor share counts loaded    23431
dtype: int64

## The putthrough defect, measured

A day's `putthrough_volume` repeating the previous day's figure exactly, at a non-zero
value, is not coincidence when it happens thousands of times: it is the field being carried
forward instead of reset. `pv_repeat_vol_pct` is the share of the year's total volume
sitting on those repeated bars — volume with no trade behind it. PGD 2017 is the clearest
case: one real block of 22,409,757 shares on 25 July, then the same figure on every session
after it to the year end, which turns a 12% annual turnover into 1,606%.

`deal_volume` is checked the same way as a control. Its repeats are orders of magnitude
smaller in volume terms and are what genuine coincidence looks like — an illiquid stock
trading the same round lot twice running.

In [3]:
PUTTHROUGH_REPEAT_SQL = """
with bars as (
    select symbol,
           date,
           total_volume,
           putthrough_volume,
           deal_volume,
           lag(putthrough_volume) over (partition by symbol order by date) as pv_prev,
           lag(deal_volume)       over (partition by symbol order by date) as dv_prev
    from fireant_prices
    where unit = 1000
)
select cast(substr(date, 1, 4) as integer)                           as year,
       count(*)                                                      as bars,
       sum(putthrough_volume > 0 and putthrough_volume = pv_prev)     as pv_repeat_bars,
       sum(deal_volume > 0 and deal_volume = dv_prev)                 as dv_repeat_bars,
       round(100.0 * sum(case when putthrough_volume > 0 and putthrough_volume = pv_prev
                              then putthrough_volume else 0 end)
             / nullif(sum(total_volume), 0), 2)                       as pv_repeat_vol_pct
from bars
group by year
order by year
"""

with closing(sqlite3.connect(DB)) as conn:
    putthrough_repeat = pd.read_sql(PUTTHROUGH_REPEAT_SQL, conn).set_index("year")

putthrough_repeat

,bars,pv_repeat_bars,dv_repeat_bars,pv_repeat_vol_pct
year,,,,
2009,89332,0,562,0.00
2010,141704,0,1260,0.00
2011,181008,116,3381,0.16
2012,191953,122,3499,0.10
2013,193766,122,2475,0.03
2014,190999,134,2258,0.04
2015,206245,122,2455,0.06
2016,239698,121,2592,0.05
2017,307875,15738,3169,13.42


## The window and the market calendar

`WINDOW` is written once and used by both queries, so the calendar and the volumes can
never drift apart. It is expressed as SQLite date arithmetic on the formation date rather
than as a hardcoded "June", so changing `WINDOW_DAYS` or `FORMATION_MMDD` moves both.

The **market calendar** is the union of every date any symbol has a bar on — indices
included, since the question is only "was the exchange open". It comes out at 20-22
sessions per window, and it is the denominator of `traded_pct`. Using it rather than the
stock's own bar count is what makes the ratio catch suspensions and part-window listings:
a stock with 6 bars in a 22-session window scores 27% at best, however busy those 6 were.

In [4]:
WINDOW = f"""
    date <= substr(date, 1, 4) || '-{FORMATION_MMDD}'
    and date > date(substr(date, 1, 4) || '-{FORMATION_MMDD}', '-{WINDOW_DAYS} days')
"""

CALENDAR_SQL = f"""
select cast(substr(date, 1, 4) as integer) as formation_year,
       count(distinct date)                as market_sessions,
       min(date)                           as window_start,
       max(date)                           as window_end
from fireant_prices
where {WINDOW}
group by formation_year
order by formation_year
"""

WINDOW_SQL = f"""
select symbol,
       cast(substr(date, 1, 4) as integer)      as formation_year,
       count(*)                                 as bars,
       sum(deal_volume > 0)                     as traded_sessions,
       sum(deal_volume)                         as deal_volume,
       sum(total_volume)                        as total_volume,
       sum(deal_volume * price_average * unit)  as deal_value_vnd,
       max(date)                                as last_bar
from fireant_prices
where unit = 1000 and {WINDOW}
group by symbol, formation_year
"""

with closing(sqlite3.connect(DB)) as conn:
    calendar = pd.read_sql(CALENDAR_SQL, conn).set_index("formation_year")
    window = pd.read_sql(WINDOW_SQL, conn)

calendar

,market_sessions,window_start,window_end
formation_year,,,
2009,22,2009-06-01,2009-06-30
2010,22,2010-06-01,2010-06-30
2011,22,2011-06-01,2011-06-30
2012,21,2012-06-01,2012-06-29
2013,20,2013-06-03,2013-06-28
2014,21,2014-06-02,2014-06-30
2015,22,2015-06-01,2015-06-30
2016,22,2016-06-01,2016-06-30
2017,22,2017-06-01,2017-06-30


In [5]:
# One row per (symbol, formation_year), then re-keyed onto the fiscal period it screens.
window = window.join(calendar["market_sessions"], on="formation_year")
window["traded_pct"] = (100 * window["traded_sessions"] / window["market_sessions"]).round(2)
window["period"] = window["formation_year"] - FORMATION_LAG_YEARS
window = window.set_index(["symbol", "period"]).sort_index()
window

formation_year  bars  ...  market_sessions  traded_pct
symbol period                        ...                             
A32    2018              2019    20  ...               20       30.00
       2019              2020    22  ...               22       22.73
       2020              2021    22  ...               22       77.27
       2021              2022    22  ...               22       40.91
       2022              2023    22  ...               22       40.91
...                       ...   ...  ...              ...         ...
YTC    2021              2022    22  ...               22        4.55
       2022              2023    22  ...               22        4.55
       2023              2024    20  ...               20       55.00
       2024              2025    21  ...               21       57.14
       2025              2026    22  ...               22        9.09

[20568 rows x 9 columns]

## The ratio

The share count comes from the row's own `period`; every volume column comes from the June
window `FORMATION_LAG_YEARS` later. `attach` is applied to both universes so the panel and
the cross-section the percentile is read off are computed by the same code, not by two
expressions that have to be kept in agreement by hand.

`turnover` divides by `shares`, the traded count. `turnover_issued` divides by
`shares_issued` and is carried, unused by the filter, so the effect of the switch can be
measured downstream instead of reconstructed.

In [6]:
WINDOW_COLS = ["formation_year", "market_sessions", "bars", "traded_sessions", "traded_pct",
               "deal_volume", "total_volume", "deal_value_vnd", "last_bar"]


def attach(capital: pd.DataFrame, window: pd.DataFrame, vendor: pd.Series,
           par_value: int = PAR_VALUE_VND) -> pd.DataFrame:
    """Window volumes joined to the share count published as of the formation date."""
    out = share_count(capital["paid_in_capital"], vendor, par_value=par_value)
    for col in WINDOW_COLS:
        out[col] = window[col].reindex(out.index)

    denom = out["shares"].replace(0, np.nan)
    issued = out["shares_issued"].replace(0, np.nan)
    out["turnover"] = out["deal_volume"] / denom
    out["turnover_issued"] = out["deal_volume"] / issued
    out["turnover_total"] = out["total_volume"] / denom
    out["putthrough_pct"] = (100 * (1 - out["deal_volume"]
                                    / out["total_volume"].replace(0, np.nan))).round(2)
    out["adtv_vnd"] = out["deal_value_vnd"] / out["traded_sessions"].replace(0, np.nan)
    return out


market_turnover = attach(market, window, vendor_shares)
turnover = attach(panel, window, vendor_shares)
turnover["has_treasury"] = panel["treasury_stock"].fillna(0).ne(0)
turnover

shares_issued  shares_out  ...      adtv_vnd has_treasury
symbol period                             ...                           
A32    2017        6800000.0   6800000.0  ...           NaN        False
       2018        6800000.0   6800000.0  ...  1.390833e+07        False
       2019        6800000.0   6800000.0  ...  2.750400e+07        False
       2020        6800000.0   6800000.0  ...  9.924225e+07        False
       2021        6800000.0   6800000.0  ...  3.309667e+07        False
...                      ...         ...  ...           ...          ...
YTC    2021        3080000.0   3080000.0  ...  1.310000e+07        False
       2022        3080000.0   3080000.0  ...  5.900000e+06        False
       2023        3080000.0   3080000.0  ...  3.002818e+07        False
       2024        9548000.0   9548000.0  ...  8.407500e+06        False
       2025        9548000.0   9548000.0  ...  2.775000e+06        False

[21233 rows x 20 columns]

## The cut

The threshold is the market's own `TURNOVER_PERCENTILE`-th percentile, recomputed for each
formation year — a relative cut, so it does not need re-tuning as the exchange's liquidity
changes decade to decade.

**The comparison is `>`, not `>=`, and that matters here.** From formation year 2017 onward
more than a tenth of the market has *zero* volume in the window, so the 10th percentile is
exactly 0, and `turnover >= 0` would admit every dead stock — the opposite of what the cut
is for. With `>`, the rule reads as intended: drop the bottom decile, and when the bottom
decile is a block of zeros, drop the whole block. That costs 10-11% of rows in the early
years and up to 22% in 2017-2019, which is the size of the zero block, not a bug.

In [7]:
min_turnover = (market_turnover.groupby("formation_year")["turnover"]
                .quantile(TURNOVER_PERCENTILE / 100)
                .rename("min_turnover"))

pd.DataFrame({
    "market_rows": market_turnover.groupby("formation_year")["turnover"].count(),
    "market_zero_turnover": market_turnover.assign(z=market_turnover["turnover"].eq(0))
                                           .groupby("formation_year")["z"].sum(),
    f"p{TURNOVER_PERCENTILE}_market": min_turnover.round(6),
    f"p{TURNOVER_PERCENTILE}_panel": (turnover.groupby("formation_year")["turnover"]
                                     .quantile(TURNOVER_PERCENTILE / 100).round(6)),
})

,market_rows,market_zero_turnover,p10_market,p10_panel
formation_year,,,,
2010.0,571,27,0.004190,0.005005
2011.0,731,31,0.000400,0.000382
2012.0,770,43,0.000114,0.000105
2013.0,776,62,0.000048,0.000049
2014.0,763,54,0.000078,0.000067
2015.0,818,48,0.000140,0.000123
2016.0,940,76,0.000038,0.000037
2017.0,1221,173,0.000000,0.000000
2018.0,1410,277,0.000000,0.000000


In [8]:
turnover = turnover.join(min_turnover, on="formation_year")

has_turnover = turnover["turnover"].notna()
turnover["tradeable"] = (
    (turnover["turnover"] > turnover["min_turnover"])
    & (turnover["traded_pct"] >= MIN_TRADED_PCT)
).astype("boolean").where(has_turnover)

turnover[["shares_issued", "shares", "shares_source", "formation_year", "market_sessions",
          "bars", "traded_sessions", "traded_pct", "deal_volume", "turnover",
          "min_turnover", "tradeable"]]

shares_issued     shares  ... min_turnover  tradeable
symbol period                            ...                        
A32    2017        6800000.0  6800000.0  ...          NaN       <NA>
       2018        6800000.0  6800000.0  ...          0.0      False
       2019        6800000.0  6800000.0  ...          0.0      False
       2020        6800000.0  6800000.0  ...          0.0       True
       2021        6800000.0  6800000.0  ...          0.0      False
...                      ...        ...  ...          ...        ...
YTC    2021        3080000.0  3080000.0  ...          0.0      False
       2022        3080000.0  3080000.0  ...          0.0      False
       2023        3080000.0  3080000.0  ...          0.0       True
       2024        9548000.0  9548000.0  ...          0.0       True
       2025        9548000.0  9548000.0  ...          0.0      False

[21233 rows x 12 columns]

### Coverage and diagnostics

In [9]:
zero_volume = has_turnover & turnover["deal_volume"].eq(0)

# Charter capital raised between the balance-sheet date and formation: the denominator is
# the last published figure, so turnover reads high wherever this is set. Flagged, not
# patched — patching it would put a number into the screen that was not public on 30 June.
next_capital = panel["paid_in_capital"].groupby(level="symbol", sort=False).shift(-1)
raised_before_formation = (next_capital / panel["paid_in_capital"]).gt(1.05).fillna(False)

pd.Series({
    "panel rows": len(turnover),
    "with a window": int(has_turnover.sum()),
    "  no bars in the window at all": int(turnover["bars"].isna().sum()),
    "  no charter capital": int((turnover["shares_issued"] == 0).sum()),
    "denominator from ShareAtPeriodEnd": int(turnover["shares_source"].eq("outstanding").sum()),
    "  fell back to par-implied": int(turnover["shares_source"].eq("par_implied").sum()),
    "bars in the window but zero volume": int(zero_volume.sum()),
    "  of those, traded_pct >= MIN_TRADED_PCT": int((zero_volume
                                                     & turnover["traded_pct"].ge(MIN_TRADED_PCT)).sum()),
    "listed under half the window": int((turnover["bars"]
                                         < turnover["market_sessions"] / 2).sum()),
    "capital raised >5% by the next balance sheet": int(raised_before_formation.sum()),
    "treasury stock held": int(turnover["has_treasury"].sum()),
})

panel rows                                      21233
with a window                                   18542
  no bars in the window at all                   2689
  no charter capital                                5
denominator from ShareAtPeriodEnd               20415
  fell back to par-implied                        818
bars in the window but zero volume               2301
  of those, traded_pct >= MIN_TRADED_PCT            0
listed under half the window                       88
capital raised >5% by the next balance sheet     3171
treasury stock held                              5109
dtype: int64

In [10]:
turnover[["turnover", "turnover_issued", "turnover_total", "traded_pct", "bars",
          "adtv_vnd"]].describe(percentiles=[.05, .10, .25, .50, .75, .90]).round(5)

,turnover,turnover_issued,turnover_total,traded_pct,bars,adtv_vnd
count,18542.00000,18542.00000,18542.00000,18544.00000,18544.00000,1.624200e+04
mean,0.05159,0.05132,0.05929,61.53791,21.28009,6.105139e+09
std,0.14307,0.14249,0.15446,39.64582,1.43445,3.494339e+10
min,0.00000,0.00000,0.00000,0.00000,1.00000,3.000000e+04
5%,0.00000,0.00000,0.00000,0.00000,20.00000,1.835023e+06
10%,0.00000,0.00000,0.00000,0.00000,20.00000,4.277583e+06
25%,0.00035,0.00035,0.00046,19.05000,21.00000,1.667189e+07
50%,0.00430,0.00427,0.00574,77.27000,22.00000,8.130313e+07
75%,0.03333,0.03301,0.04307,100.00000,22.00000,8.240585e+08
90%,0.14236,0.14164,0.16826,100.00000,22.00000,7.254225e+09


In [11]:
# By fiscal period. `min_turnover` is the cut that period's rows actually faced, and it
# going to zero from 2016 on is the market's bottom decile becoming untraded, not a defect.
kept = turnover["tradeable"].fillna(False)

pd.DataFrame({
    "formation": turnover.groupby(level="period")["formation_year"].first(),
    "rows": turnover.groupby(level="period").size(),
    "with_window": has_turnover.groupby(level="period").sum(),
    "min_turnover": turnover.groupby(level="period")["min_turnover"].first().round(6),
    "med_turnover": turnover["turnover"].groupby(level="period").median().round(5),
    "med_traded_pct": turnover["traded_pct"].groupby(level="period").median(),
    "tradeable": kept.groupby(level="period").sum(),
    "kept_pct": (100 * kept.groupby(level="period").sum()
                 / has_turnover.groupby(level="period").sum()).round(1),
})

,formation,rows,with_window,min_turnover,med_turnover,med_traded_pct,tradeable,kept_pct
period,,,,,,,,
2009,2010.0,525,342,0.004190,0.09882,100.00,303,88.6
2010,2011.0,751,591,0.000400,0.01788,95.45,483,81.7
2011,2012.0,875,686,0.000114,0.01074,95.24,522,76.1
2012,2013.0,984,731,0.000048,0.00905,85.00,541,74.0
2013,2014.0,1060,722,0.000078,0.00593,85.71,494,68.4
2014,2015.0,1111,751,0.000140,0.00784,86.36,541,72.0
2015,2016.0,1255,879,0.000038,0.00878,86.36,591,67.2
2016,2017.0,1369,1120,0.000000,0.00443,72.73,672,60.0
2017,2018.0,1482,1350,0.000000,0.00135,42.86,637,47.2


In [12]:
# Which screen is doing the work. The percentile cut removes almost nothing that the
# traded_pct floor has not already removed — every zero-volume row has traded_pct = 0, so
# the two overlap almost completely at a 50% floor. The turnover screen only starts to
# bind on its own once the floor is dropped.
passes_turnover = has_turnover & (turnover["turnover"] > turnover["min_turnover"])

pd.DataFrame([{
    "traded_pct >=": y,
    "traded_pct only": int((has_turnover & turnover["traded_pct"].ge(y)).sum()),
    "both screens": int((passes_turnover & turnover["traded_pct"].ge(y)).sum()),
    "turnover removes extra": int((has_turnover & turnover["traded_pct"].ge(y)
                                   & ~passes_turnover).sum()),
} for y in [0, 10, 25, 50, 75, 90]]).set_index("traded_pct >=")

,traded_pct only,both screens,turnover removes extra
traded_pct >=,,,
0,18542,16070,2472
10,15027,14948,79
25,13262,13220,42
50,11469,11444,25
75,9498,9482,16
90,8142,8132,10


In [13]:
# The case the filter exists for: bars on essentially every session of the window, and not
# one trade in any of them.
turnover.loc[zero_volume & turnover["bars"].ge(turnover["market_sessions"]),
             ["formation_year", "bars", "market_sessions", "traded_sessions",
              "total_volume", "turnover"]].head(10)

formation_year  bars  ...  total_volume  turnover
symbol period                        ...                        
AC4    2018            2019.0  20.0  ...           0.0       0.0
       2019            2020.0  22.0  ...           0.0       0.0
ACS    2018            2019.0  20.0  ...           0.0       0.0
AFC    2018            2019.0  20.0  ...           0.0       0.0
AMP    2019            2020.0  22.0  ...           0.0       0.0
AMV    2024            2025.0  21.0  ...           0.0       0.0
ANT    2017            2018.0  21.0  ...        4600.0       0.0
APL    2016            2017.0  22.0  ...           0.0       0.0
       2017            2018.0  21.0  ...           0.0       0.0
       2018            2019.0  20.0  ...           0.0       0.0

[10 rows x 6 columns]

### What the filter costs the strategy

The thresholds are only meaningful against the selection they act on. This joins
`f_score_panel.csv` as it stands on disk — the overlap is printed first, because a panel
regenerated after the F-score notebook last ran would make the counts below understate.

In [14]:
CUTOFF = 8   # the F-score cutoff in use

f_score_panel = pd.read_csv(f"{RESULTS}/f_score_panel.csv").set_index(["symbol", "period"])
print(f"f_score_panel rows: {len(f_score_panel)}, "
      f"of which in this panel: {int(f_score_panel.index.isin(turnover.index).sum())}")

selected = f_score_panel["f_score"] >= CUTOFF
tradeable = turnover["tradeable"].reindex(f_score_panel.index)

cost = pd.DataFrame({
    "selected": selected.groupby(level="period").sum(),
    "tradeable": (selected & tradeable.fillna(False)).groupby(level="period").sum(),
    "dropped": (selected & (tradeable == False)).groupby(level="period").sum(),  # noqa: E712
    "no_window": (selected & tradeable.isna()).groupby(level="period").sum(),
})
cost.loc["ALL"] = cost.sum()
cost = cost.astype(int)
cost["kept_pct"] = (100 * cost["tradeable"] / cost["selected"]).round(1)
cost

f_score_panel rows: 21233, of which in this panel: 21233


,selected,tradeable,dropped,no_window,kept_pct
period,,,,,
2009,0,0,0,0,NaN
2010,0,0,0,0,NaN
2011,35,27,5,3,77.1
2012,50,29,11,10,58.0
2013,97,52,23,22,53.6
2014,121,72,23,26,59.5
2015,103,53,32,18,51.5
2016,121,71,41,9,58.7
2017,126,62,61,3,49.2


### What the denominator switch changed

The filter's output is a boolean, so the question is not how much `turnover` moved but how
many verdicts flipped. Netting treasury out raises a stock's turnover — but it raises the
market's cross-section too, and the cut is a percentile of that cross-section. The
counterfactual therefore has to move both: threshold recomputed on the issued-share basis,
compared against the issued-share turnover. Holding the new threshold fixed would measure
only half the change, and would make the switch look strictly favourable when it is not.

In [15]:
min_turnover_issued = (market_turnover.groupby("formation_year")["turnover_issued"]
                       .quantile(TURNOVER_PERCENTILE / 100)
                       .rename("min_turnover_issued"))
cut_issued = turnover["formation_year"].map(min_turnover_issued)

tradeable_issued = ((turnover["turnover_issued"] > cut_issued)
                    & turnover["traded_pct"].ge(MIN_TRADED_PCT)
                    ).astype("boolean").where(has_turnover)

now, before = turnover["tradeable"].fillna(False), tradeable_issued.fillna(False)
lift = (turnover["turnover"] / turnover["turnover_issued"]).pipe(lambda s: s[s.gt(1.0001)])

pd.Series({
    "rows with a verdict": int(has_turnover.sum()),
    "rows whose turnover rose": f"{len(lift)} (median {lift.median():.4f}x)",
    "tradeable, issued-share basis": int(before.sum()),
    "tradeable, outstanding basis": int(now.sum()),
    "flipped to tradeable": int((now & ~before).sum()),
    "flipped to untradeable": int((before & ~now).sum()),
})

rows with a verdict                              18542
rows whose turnover rose         3246 (median 1.0121x)
tradeable, issued-share basis                    11445
tradeable, outstanding basis                     11444
flipped to tradeable                                 0
flipped to untradeable                               1
dtype: object

In [16]:
# The same cost at other floors. `no_window` rows are never tradeable, at any threshold.
pd.DataFrame([{
    "traded_pct >=": y,
    "tradeable at F>=CUTOFF": int((selected & (passes_turnover
                                               & turnover["traded_pct"].ge(y)
                                               ).reindex(f_score_panel.index).fillna(False)).sum()),
} for y in [0, 10, 25, 50, 75]]).set_index("traded_pct >=")

,tradeable at F>=CUTOFF
traded_pct >=,
0,1613
10,1500
25,1334
50,1142
75,920


### Export

Keyed on `(symbol, period)`, so it joins straight onto `f_score_panel.csv` and
`book_to_market_panel.csv`. `formation_year`, `min_turnover` and the session counts travel
with the verdict: a `tradeable` boolean on its own cannot be argued with three notebooks
downstream. The five `share*` columns are the same five the BM panel carries, on the same
key and from the same call, so the join downstream drops one copy and asserts they match.

In [17]:
turnover.to_csv(f"{RESULTS}/trade_turnover_panel.csv")
turnover.shape

(21233, 22)